In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os 

In [ ]:
IN_DIR = "../data/raw"
IN_FILE = "sample.png"
OUT_DIR = "../data/processed"
OUT_FILE = "roi_overlay.png"
IN_PATH = os.path.join(IN_DIR, IN_FILE)
OUT_PATH = os.path.join(OUT_DIR, OUT_FILE)

In [ ]:
drawing = False
ix , iy = -1, -1

In [ ]:
blue_row_tops = [255, 335, 418, 500, 580] # (y1, y1 + row_h)
red_row_tops = [815, 895, 980, 1065, 1145] # (y2, y2 + row_h)
row_h = 80

name_x = (535, 915) # (x1, x2)
col_x = {
    'E': (915, 995), #( x1, x2)
    'A': (995, 1065),
    'D': (1065, 1135),
    'DMG': (1135, 1280),
    'H': (1280, 1425),
    'MIT': (1425, 1547)
}

In [ ]:
def show(img, figsize=(16, 9)):
    """Display a BGR OpenCV image inline (matplotlib expects RGB)."""
    fig, ax = plt.subplots(figsize=figsize)
    # ax.minorticks_on()   
    ax.set_xticks([535, 915, 995, 1065, 1135, 1280, 1425, 1547])
    ax.set_yticks([255, 335, 418, 500, 580, 665, 815, 895, 980, 1065, 1145, 1226]) # X = X + (~85)
    ax.grid(True, which="both", axis="both", color="lime", linewidth=0.5, alpha=0.5)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    fig.savefig("../data/processed/roi_template.png")
    return fig, ax

In [ ]:
def draw_rois(img, row_tops, row_h, name_x, col_x):
    img_copy = img.copy()
    for top in row_tops:
        cv2.rectangle(img_copy, (name_x[0], top), (name_x[1], top + row_h), (0, 255, 0), 2, 0, None)
        for (x1, x2) in col_x.values():
            cv2.rectangle(img_copy, (x1, top), (x2, top + row_h), (0, 255, 0), 2, 0, None)
    return img_copy

In [ ]:
img = cv2.imread(IN_PATH)
H, W = img.shape[:2]
print("Image size:", W, 'x', H)

In [ ]:
fig, ax = show(img)

In [ ]:
vis = draw_rois(img, blue_row_tops, row_h, name_x, col_x)
vis = draw_rois(vis, red_row_tops, row_h, name_x, col_x)
show(vis)

In [ ]:
cv2.imwrite(OUT_PATH, vis)

In [ ]:
import json

In [ ]:
CONFIG_DIR = "../configs"
JSON_FILE = "template.json"
CONFIG_PATH = os.path.join(CONFIG_DIR, JSON_FILE)

In [ ]:
config = {
    "template_name": CONFIG_PATH,
    "image_size": [W, H],
    "row_h": row_h,
    "teams": {
        "blue": {"row_tops": blue_row_tops},
        "red": {"row_tops": red_row_tops}
    },
    "name_x": list(name_x),
    "col_x": {
        k: list(v) for k, v in col_x.items()
    },
}

In [ ]:
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)
print(f"saved {CONFIG_PATH}")